In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def estg_phase(f, alpha_M, Mc, f_min=1e-5):
    G = 6.67430e-11
    c = 299792458.0
    factor = (5.0 / 192.0) * (G * Mc / c**3)**(-5.0/3.0)
    integral = (f_min**(-5.0/3.0) - f**(-5.0/3.0))
    return alpha_M * factor * integral

def snr_estg(f, h_gr, alpha_M, Mc, Sn, f_min=1e-5):
    delta_psi = estg_phase(f, alpha_M, Mc, f_min)
    h_estg = h_gr * np.exp(1j * delta_psi)
    snr2 = 4 * np.trapz(np.abs(h_estg)**2 / Sn, f)
    return np.sqrt(snr2)

def fisher_alpha_M(f, h_gr, alpha_M, Mc, Sn, f_min=1e-5):
    G = 6.67430e-11
    c = 299792458.0
    prefactor = (5.0 / 192.0) * (G * Mc / c**3)**(-5.0/3.0)
    dpsi_dalpha = prefactor * (f_min**(-5.0/3.0) - f**(-5.0/3.0))
    F = 4 * np.trapz((dpsi_dalpha * np.abs(h_gr))**2 / Sn, f)
    return np.sqrt(1.0 / F)

def lisa_noise(f):
    return 1e-41 * (1 + (1e-4 / f)**2)

f = np.logspace(-5, -1, 1000)
Sn = lisa_noise(f)
G = 6.67430e-11
c = 299792458.0
M_sun = 1.98847e30
M1 = 1e6 * M_sun
M2 = 1e6 * M_sun
Mc = (M1 * M2)**(3/5) / (M1 + M2)**(1/5)
h_gr = 1e-22 * (f / 1e-3)**(-2/3)

alpha_M = 0.0
snr = snr_estg(f, h_gr, alpha_M, Mc, Sn)
sigma = fisher_alpha_M(f, h_gr, alpha_M, Mc, Sn)

print(f"SNR (alpha_M=0): {snr:.1f}")
print(f"sigma(alpha_M): {sigma:.2e}")

plt.figure(figsize=(10,6))
plt.loglog(f, np.abs(estg_phase(f, 0.001, Mc)), label='alpha_M=0.001')
plt.loglog(f, np.abs(estg_phase(f, 0.01, Mc)), label='alpha_M=0.01')
plt.xlabel('f [Hz]')
plt.ylabel('|Delta Psi| [rad]')
plt.legend()
plt.grid(True)
plt.savefig('lisa_sensitivity.png')
plt.show()

print("\n--- SONUCLAR ---")
print(f"SNR (alpha_M=0): {snr:.1f}")
print(f"sigma(alpha_M): {sigma:.2e}")